# Making predictions and storing in a CSV

## Import Libraries

In [2]:
import os
import sys
import torch
import pandas as pd
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from pathlib import Path
from tqdm import tqdm

sys.path.append("../")
from utils import image_models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

c:\Users\gnoceras\Documents\HackIL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


## Load Model

In [3]:
# Model configuration
num_classes = 6  # Adjust based on your dataset
model_name = "mobilenetv3_small_100"
model_path = "../models/MobileNetv3.pth"

# Load model
model = image_models.MobileNetV3(
    model_name=model_name,
    num_classes=num_classes
).to(device)

model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print(f"Model loaded from {model_path}")

Model loaded from ../models/MobileNetv3.pth


## Load Dataset

In [4]:
# Data directory
data_dir = "../data/Nutrition_dataset"

# Transforms (same as used during training)
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Load both train and test datasets
train_dataset = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=transform)
val_dataset = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=transform)
test_dataset = datasets.ImageFolder(os.path.join(data_dir, "test"), transform=transform)

# Get class names
class_names = train_dataset.classes
print(f"Classes: {class_names}")
print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Classes: ['ALL Present', 'ALLAB', 'KAB', 'NAB', 'PAB', 'ZNAB']
Train samples: 7056
Validation samples: 882
Test samples: 882


## Make Predictions on All Images

In [6]:
def predict_dataset(model, dataset, dataset_name, device):
    """
    Predict all images in a dataset and return results as a list of dicts
    """
    results = []
    
    with torch.no_grad():
        for idx in tqdm(range(len(dataset)), desc=f"Processing {dataset_name}"):
            # Get image and true label
            image, true_label_idx = dataset[idx]
            image_path, _ = dataset.imgs[idx]
            
            # Convert to relative path
            rel_path = os.path.relpath(image_path, start=data_dir)
            
            # Predict
            image_batch = image.unsqueeze(0).to(device)
            
            with torch.amp.autocast("cuda" if torch.cuda.is_available() else "cpu"):
                logits = model(image_batch)
            
            pred_label_idx = torch.argmax(logits, dim=1).item()
            confidence = torch.softmax(logits, dim=1)[0, pred_label_idx].item()
            
            # Store result
            results.append({
                'image_path': rel_path,
                'true_class': class_names[true_label_idx],
                'predicted_class': class_names[pred_label_idx],
                'confidence': confidence,
                'correct': true_label_idx == pred_label_idx,
                'dataset': dataset_name
            })
    
    return results

# Process both datasets
train_results = predict_dataset(model, train_dataset, "train", device)
val_results = predict_dataset(model, val_dataset, "val", device)
test_results = predict_dataset(model, test_dataset, "test", device)

# Combine results
all_results = train_results + val_results + test_results

# Create DataFrame
df_predictions = pd.DataFrame(all_results)
print(f"\\nTotal predictions: {len(df_predictions)}")
print(f"Overall accuracy: {df_predictions['correct'].mean():.4f}")
df_predictions.head(10)

Processing test: 100%|██████████| 882/882 [00:20<00:00, 43.64it/s]

\nTotal predictions: 8820
Overall accuracy: 0.9879


,image_path,true_class,predicted_class,confidence,correct,dataset
0,train\ALL Present\0412_1.jpg,ALL Present,ALL Present,0.999512,True,train
1,train\ALL Present\0412_3.jpg,ALL Present,ALL Present,1.000000,True,train
2,train\ALL Present\0412_4.jpg,ALL Present,ALL Present,1.000000,True,train
3,train\ALL Present\0413_0.jpg,ALL Present,ALL Present,0.999512,True,train
4,train\ALL Present\0413_1.jpg,ALL Present,ALL Present,0.999023,True,train
5,train\ALL Present\0413_2.jpg,ALL Present,ALL Present,1.000000,True,train
6,train\ALL Present\0413_3.jpg,ALL Present,ALL Present,1.000000,True,train
7,train\ALL Present\0413_4.jpg,ALL Present,ALL Present,1.000000,True,train
8,train\ALL Present\0414_0.jpg,ALL Present,ALL Present,0.999512,True,train
9,train\ALL Present\0414_1.jpg,ALL Present,ALL Present,0.999512,True,train


## Summary Statistics

In [7]:
# Accuracy by dataset
print("Accuracy by Dataset:")
print(df_predictions.groupby('dataset')['correct'].mean())
print()

# Accuracy by true class
print("Accuracy by True Class:")
print(df_predictions.groupby('true_class')['correct'].agg(['mean', 'count']))
print()

# Average confidence
print(f"Average confidence: {df_predictions['confidence'].mean():.4f}")
print(f"Average confidence (correct): {df_predictions[df_predictions['correct']]['confidence'].mean():.4f}")
print(f"Average confidence (incorrect): {df_predictions[~df_predictions['correct']]['confidence'].mean():.4f}")

Accuracy by Dataset:
dataset
test     0.985261
train    0.987812
val      0.990930
Name: correct, dtype: float64

Accuracy by True Class:
                 mean  count
true_class                  
ALL Present  0.970748   1470
ALLAB        1.000000   1470
KAB          0.999320   1470
NAB          1.000000   1470
PAB          0.985714   1470
ZNAB         0.971429   1470

Average confidence: 0.9899
Average confidence (correct): 0.9930
Average confidence (incorrect): 0.7426


## Save Predictions to CSV

In [8]:
# Save to CSV
output_path = "../data/processed/predictions/proximal_sensing.csv"
df_predictions.to_csv(output_path, index=False)
print(f"Predictions saved to {output_path}")

Predictions saved to ../data/processed/predictions/proximal_sensing.csv
